# ED Alignment Algorithm on Real Piano Performances

This notebook evaluates the ED alignment algorithm in `compareMusic` on real piano performances from the [(n)ASAP: the (note-)Aligned Scores And Performances dataset](https://github.com/CPJKU/asap-dataset) (Peter et al., 2023).

(n)ASAP is a dataset of aligned musical scores and performances built by extending the ASAP dataset with note-level annotations. The ASAP contains 236 distinct musical scores and 1067 performances of Western classical piano music from 15 different composers, the piece directory contains the XML and MIDI score, plus all of the performances of a specific piece, including: 
- `midi_score`: the MIDI score (what should be played) — used as **reference**
- `midi_performance`: what the pianist actually played — used as **response**
- `note_alignments.tsv`: official note-level alignment annotations — used as **ground truth**

bib citation:

@article{Peter-2023,
 title = {Automatic Note-Level Score-to-Performance Alignments in the ASAP Dataset},
 author = {Peter, Silvan David and Cancino-Chacón, Carlos Eduardo and Foscarin, Francesco and McLeod, Andrew Philip and Henkel, Florian and Karystinaios, Emmanouil and Widmer, Gerhard},
 doi = {10.5334/tismir.149},
 journal = {Transactions of the International Society for Music Information Retrieval {(TISMIR)}},
 year = {2023}
}

relevant paper: https://transactions.ismir.net/articles/10.5334/tismir.149#5-alignment-of-the-asap-dataset

## Download the ASAP Dataset

In [19]:
import os

# Note: use CPJKU version (not fosfrancesco) because only CPJKU has
# the note_alignments TSV files are needed for ground truth comparison.
if not os.path.exists("asap-dataset"):
    os.system("git clone https://github.com/CPJKU/asap-dataset.git")
else:
    print("asap-dataset already exists, skipping download.")

ASAP_PATH = "asap-dataset"

asap-dataset already exists, skipping download.


## Load the ground truth

The `note_alignment.tsv` file in each performance contains the official note-level alignment annotations. Each row in the TSV represents one note pair. 

The `label` has three possible values:
- `match`: a score note (reference) and a performance note (response) were successfully aligned
- `deletion`: the score note was not played in the performance
- `insertion`: an extra note was played that has no counterpart in the score

For `match` and `insertion` rows, the TSV provides `onset` (onset time in seconds) and `pitch` (MIDI pitch) for the performance note.

In [20]:
import csv

def load_ground_truth(tsv_path):
    """
    Read a note_alignments TSV file from the CPJKU ASAP dataset.

    The TSV columns are: xml_id, midi_id, track, channel, pitch, onset
        - if xml_id == 'insertion': extra note played with no score counterpart
        - if midi_id == 'deletion': score note not played in the performance
        - otherwise: matched pair, onset and pitch refer to the performance note

    Returns a list of dicts with keys: label, onset, pitch.
    """
    rows = []
    with open(tsv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f, delimiter="\t")
        for row in reader:
            xml_id  = row["xml_id"].strip()
            midi_id = row["midi_id"].strip()

            if midi_id == "deletion":
                label = "deletion"
                onset = None
                pitch = None
            elif xml_id == "insertion":
                label = "insertion"
                onset = float(row["onset"])
                pitch = int(row["pitch"])
            else:
                label = "paired"
                onset = float(row["onset"])
                pitch = int(row["pitch"])

            rows.append({
                "label": label, # "paired", "insertion", or "deletion"
                "onset": onset,
                "pitch": pitch,
            })
    return rows

## Load Score (reference)/Performance (response) pairs from ASAP

Helper functions for format conversion: These functions convert a MIDI file into the `{pitch, start, duration}` format used by `compare_MIDI.py`.

In [21]:
import pretty_midi

def midi_file_to_notes(midi_path):
    """
    Parse a MIDI file and return all its notes in compareMusic format.
    Notes are sorted by onset time.
    """
    midi_data = pretty_midi.PrettyMIDI(midi_path)
    all_notes = []
    for instrument in midi_data.instruments:
        if instrument.is_drum: # Ignore drum tracks
            continue
        for note in instrument.notes:
            all_notes.append({
                "pitch": note.pitch,
                "start": round(note.start, 3),
                "duration": round(note.end - note.start, 3),
            })
    all_notes.sort(key=lambda n: (n["start"], n["pitch"]))
    return all_notes


def build_sample(ref_path, response_path, composer, title, metadata_row):
    """
    Convert one score/performance MIDI pair into a sample dict
    ready for compare_performance_ED.
    Returns None if either MIDI file produces zero notes.
    """
    score_notes = midi_file_to_notes(ref_path)
    perf_notes  = midi_file_to_notes(response_path)

    if not score_notes or not perf_notes:
        return None

    return {
        "composer": composer,
        "title": title,
        "reference": {"notes": score_notes},
        "response": {"notes": perf_notes},
        "metadata_row": metadata_row,
    }

For the ASAP dataset, the `metadata.csv` lists every score/performance pair in the dataset, check that both MIDI files exist on disk.

In [22]:
def load_samples(asap_path, composer):
    """
    Read the ASAP metadata CSV and return a list of sample dicts
    for a specific composer only.
    """
    metadata_path = os.path.join(asap_path, "metadata.csv")
    samples = []

    with open(metadata_path, "r", encoding="utf-8") as csv_file:
        reader = csv.DictReader(csv_file)
        for row in reader:
            if row.get("composer", "").strip() == composer:
                ref_path = os.path.join(asap_path, row.get("midi_score", "").strip())
                response_path = os.path.join(asap_path, row.get("midi_performance", "").strip())

                if os.path.isfile(ref_path) and os.path.isfile(response_path):
                    sample = build_sample(
                        ref_path, response_path,
                        row.get("composer", "Unknown"),
                        row.get("title", "Unknown"),
                        dict(row),
                    )
                    if sample is not None:
                        samples.append(sample)

    return samples

samples = load_samples(ASAP_PATH, "Bach")

print("Loaded", len(samples), "score (reference) /performance (response) pairs.")
for i, sample in enumerate(samples[:10]):
    print(f"{str(i + 1)}. {sample['composer']}({sample['title']}) |"
          f" ref notes: {len(sample['reference']['notes'])} |"
          f" response notes: {len(sample['response']['notes'])}")

/Users/jz7125/compareMusic/.venv/lib/python3.13/site-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(


Loaded 169 score (reference) /performance (response) pairs.
1. Bach(Fugue_bwv_846) | ref notes: 755 | response notes: 754
2. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1435
3. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1438
4. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1429
5. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1431
6. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1433
7. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1438
8. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1440
9. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1431
10. Bach(Fugue_bwv_848) | ref notes: 1429 | response notes: 1430


# Run Alignment on Each Pair

Passing each score/performance pair through `compare_performance_ED`, where score MIDI is the reference and the performance MIDI is the response.

In [23]:
from evaluation_function.compare_MIDI import compare_performance_ED

def run_alignment_on_samples(samples):
    """
    Run compare_performance_ED on every sample and return the results.

    Args:
        samples: list of sample dicts from load_samples().

    Returns:
        List of result dicts, each with keys:
            composer, title, stats, event_details, is_correct
    """
    results = []
    for sample in samples:
        result = compare_performance_ED(
            sample["response"],
            sample["reference"]
        )
        results.append({
            "composer": sample["composer"],
            "title": sample["title"],
            "stats": result.stats,
            "event_details": result.event_details,
            "is_correct": result.is_correct,
        })
    return results

all_results = run_alignment_on_samples(samples)

## Compute Precision, Recall, and F1 

Compares our alignment output against the ASAP ground truth to compute standard information retrieval metrics:

- **Precision**: TP / (TP + FP) 
        - i.e. of all note pairs we called a match, how many the ground truth also call a match
- **Recall**: TP / (TP + FN)
        - i.e of all note pairs the ground truth called a match, how many we find
- **F1**: harmonic mean of precision and recall

The alignment is correct if the pitch is correct and the number of matches at that pitch does not exceed the ground truth count. Onset time is intentionally excluded from this metric, consistent with the pipeline's design: the cost function uses pitch distance only, so that timing evaluation remains a separate step performed after structural alignment rather than influencing the alignment itself.

In [33]:
def convert_pipeline_output(event_details, response_notes):
    """
    Convert event_details from compare_performance_ED into a list of
    dicts with keys: onset, pitch, label.
    Labels follow the TSV convention:
        'paired': correctly aligned note pair (pitch matches)
        'deletion' : score note not found in response (missing)
        'insertion': response note has no score counterpart (extra)

    Chord events are expanded note-by-note using correct_pitches /
    missing_pitches / extra_pitches from event_level_feedback.

    Args:
        event_details: list of dicts from compare_performance_ED
        response_notes: list of note dicts (the flat, normalised note list
                         that was passed into compare_performance_ED as
                         responseMIDI["notes"], after normalize_start_times)

    Returns:
        list of dicts with keys: onset, pitch, label
    """
    my_pairs = []

    for event in event_details:
        op = event["operation_type"]

        if event["event_type"] == "note":
            if op == "match" or op == "replacement":
                note = response_notes[event["response_index"] - 1]
                my_pairs.append({
                    "onset": note["start"],
                    "pitch": note["pitch"],
                    "label": "paired",
                })
            elif op == "missing":
                my_pairs.append({
                    "onset": None,
                    "pitch": None,
                    "label": "deletion",
                })
            elif op == "extra":
                note = response_notes[event["response_index"] - 1]
                my_pairs.append({
                    "onset": note["start"],
                    "pitch": note["pitch"],
                    "label": "insertion",
                })

        elif event["event_type"] == "chord":
            if op == "missing":
                # Entire chord missing: one deletion per note in the ref chord
                # We know how many notes because correct + missing = all ref notes
                num_ref_notes = len(event["correct_pitches"] or []) + \
                                len(event["missing_pitches"] or [])
                for _ in range(num_ref_notes):
                    my_pairs.append({
                        "onset": None,
                        "pitch": None,
                        "label": "deletion",
                    })
            elif op == "extra":
                # Entire chord is extra: one insertion per note played
                res_idx_0 = event["response_index"] - 1
                note = response_notes[res_idx_0]
                my_pairs.append({
                    "onset": note["start"],
                    "pitch": note["pitch"],
                    "label": "insertion",
                })
            else:
                # Aligned chord pair (match or replacement)
                # correct_pitches -> match for each
                # missing_pitches -> deletion for each
                # extra_pitches -> insertion for each
                res_idx_0 = event["response_index"] - 1
                note = response_notes[res_idx_0]
                onset = note["start"]
                for _ in (event["correct_pitches"] or []):
                    my_pairs.append({
                        "onset": onset,
                        "pitch": note["pitch"],
                        "label": "paired",
                    })
                for _ in (event["missing_pitches"] or []):
                    my_pairs.append({
                        "onset": None,
                        "pitch": None,
                        "label": "deletion",
                    })
                for _ in (event["extra_pitches"] or []):
                    my_pairs.append({
                        "onset": onset,
                        "pitch": note["pitch"],
                        "label": "insertion",
                    })

    return my_pairs


def compute_metrics(ground_truth, my_pairs):
    """
    Compute the metrics in a fairer way for pipelines that align 
    by pitch without considering onset time.

    For my match, count as a True Positive if:
        - its pitch appears in the GT match set for that pitch,
          and the GT has at least one unaccounted match at that pitch.

    This uses a multiset comparison: if GT has 3 matches at pitch 60
    and we predict 3 matches at pitch 60, all 3 are TPs. If we predict
    4, the extra one is an FP.

    Args:
        ground_truth: list of dicts from load_ground_truth()
        my_pairs: list of dicts from convert_pipeline_output()

    Returns:
        dict with precision, recall, f1, tp, fp, fn
    """
    # Build a pitch count multiset for GT matches
    gt_pitch_counts = {}
    for row in ground_truth:
        if row["label"] == "paired":
            p = int(row["pitch"])
            if p not in gt_pitch_counts:
                gt_pitch_counts[p] = 0
            gt_pitch_counts[p] += 1

    # Build a pitch count multiset for my matches
    my_pitch_counts = {}
    for row in my_pairs:
        if row["label"] == "paired" and row["onset"] is not None:
            p = int(row["pitch"])
            if p not in my_pitch_counts:
                my_pitch_counts[p] = 0
            my_pitch_counts[p] += 1

    tp = 0
    fp = 0
    fn = 0

    all_pitches = set(gt_pitch_counts.keys()).union(set(my_pitch_counts.keys()))
    for pitch in all_pitches:
        gt_count = gt_pitch_counts.get(pitch, 0) # Count of GT matches at this pitch
        my_count = my_pitch_counts.get(pitch, 0) # Count of my matches at this pitch
        tp += min(gt_count, my_count) 
        fp += max(0, my_count - gt_count) 
        fn += max(0, gt_count - my_count)

    if tp + fp > 0:
        precision = tp / (tp + fp)
    else:
        precision = 0.0

    if tp + fn > 0:
        recall = tp / (tp + fn)
    else:
        recall = 0.0

    if precision + recall > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = 0.0

    return {
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
        "tp": tp, "fp": fp, "fn": fn,
    }


def evaluate_one_piece(asap_path, metadata_row, event_details, response_notes_original):
    """
    Run ground truth comparison for one score/performance pair.

    Args:
        asap_path: str, path to ASAP repo root
        metadata_row: dict, one row from metadata.csv
        event_details: list from compare_performance_ED result
        response_notes_original: list of note dicts with original
                                 (absolute) onset times — NOT normalised

    Returns:
        metrics dict, or None if the TSV file is not found
    """
    tsv_rel = metadata_row.get("note_alignments", "").strip()
    tsv_path = os.path.join(asap_path, tsv_rel)

    if not os.path.isfile(tsv_path):
        print("TSV not found:", tsv_path)
        return None

    ground_truth = load_ground_truth(tsv_path)
    my_pairs = convert_pipeline_output(event_details, response_notes_original)

    return compute_metrics(ground_truth, my_pairs)

In [34]:
import pandas as pd

eval_rows = []
for sample, result in zip(samples, all_results):
    metrics = evaluate_one_piece(
        ASAP_PATH,
        sample["metadata_row"],
        result["event_details"],
        sample["response"]["notes"], 
    )
    if metrics is not None:
        eval_rows.append({
            "Piece": result["composer"] + " (" + result["title"] + ")",
            "Precision": metrics["precision"],
            "Recall": metrics["recall"],
            "F1": metrics["f1"],
            "TP": metrics["tp"],
            "FP": metrics["fp"],
            "FN": metrics["fn"],
        })

df_eval = pd.DataFrame(eval_rows)
display(df_eval.head(10))

print("Mean Precision:", round(df_eval["Precision"].mean(), 4))
print("Mean Recall :", round(df_eval["Recall"].mean(), 4))
print("Mean F1 :", round(df_eval["F1"].mean(), 4))

,Piece,Precision,Recall,F1,TP,FP,FN
0,Bach (Fugue_bwv_846),0.9092,0.8415,0.8740,621,62,117
1,Bach (Fugue_bwv_848),0.9293,0.9004,0.9146,1275,97,141
2,Bach (Fugue_bwv_848),0.9288,0.8946,0.9114,1265,97,149
3,Bach (Fugue_bwv_848),0.9420,0.9100,0.9257,1284,79,127
4,Bach (Fugue_bwv_848),0.9435,0.9147,0.9289,1287,77,120
5,Bach (Fugue_bwv_848),0.9470,0.9190,0.9328,1304,73,115
6,Bach (Fugue_bwv_848),0.9184,0.8905,0.9043,1261,112,155
7,Bach (Fugue_bwv_848),0.9394,0.9014,0.9200,1271,82,139
8,Bach (Fugue_bwv_848),0.9381,0.9096,0.9236,1288,85,128
9,Bach (Fugue_bwv_848),0.9366,0.9082,0.9222,1286,87,130


Mean Precision: 0.9238
Mean Recall : 0.8876
Mean F1 : 0.9051
